# Lab Assignment 2 - Sentiment Analysis
# Name: Mohd Adli Syukri bin Noraman
# ID: SW01083745

# Name: Muhammad Izzul Danish bin Abdul Rasib
# ID: SW01083596

Sentiment Analysis on the Amazon Fine Food Reviews dataset using:
- **Lexicon-based**: VADER
- **Machine Learning-based**: Naive Bayes

## 1. Data Preprocessing

In [18]:
import pandas as pd
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [19]:
# Load only Score and Text columns for speed
df = pd.read_csv('Reviews.csv', usecols=['Score', 'Text'])
print(f'Original dataset size: {len(df)} rows')
df.head()

Original dataset size: 568454 rows


,Score,Text
0,5,I have bought several of the Vitality canned d...
1,1,Product arrived labeled as Jumbo Salted Peanut...
2,4,This is a confection that has been around a fe...
3,2,If you are looking for the secret ingredient i...
4,5,Great taffy at a great price. There was a wid...


In [20]:
# Drop rows with missing text
df = df.dropna(subset=['Text'])

# Convert Score to sentiment labels
# Score 1-2 = negative, Score 3 = neutral, Score 4-5 = positive
def score_to_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df['sentiment'] = df['Score'].apply(score_to_sentiment)
print('Sentiment distribution (full dataset):')
print(df['sentiment'].value_counts())
df.head()

Sentiment distribution (full dataset):
sentiment
positive    443777
negative     82037
neutral      42640
Name: count, dtype: int64


,Score,Text,sentiment
0,5,I have bought several of the Vitality canned d...,positive
1,1,Product arrived labeled as Jumbo Salted Peanut...,negative
2,4,This is a confection that has been around a fe...,positive
3,2,If you are looking for the secret ingredient i...,negative
4,5,Great taffy at a great price. There was a wid...,positive


In [21]:
# Sample 5000 rows per class for balanced & fast processing
# This keeps the notebook fast while still being representative
sample_size = 30000

df_sampled = df.groupby('sentiment', group_keys=False).apply(
    lambda x: x.sample(n=min(sample_size, len(x)), random_state=42)
).reset_index(drop=True)

print(f'Sampled dataset size: {len(df_sampled)} rows')
print(df_sampled['sentiment'].value_counts())

Sampled dataset size: 90000 rows
sentiment
negative    30000
neutral     30000
positive    30000
Name: count, dtype: int64


In [22]:
# Basic text cleaning
df_sampled['clean_text'] = df_sampled['Text'].str.lower().str.replace(r'<.*?>', ' ', regex=True).str.replace(r'[^a-zA-Z\s]', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()
df_sampled.head()

,Score,Text,sentiment,clean_text
0,2,"I have an absolute passion for deep, dark hot ...",negative,i have an absolute passion for deep dark hot c...
1,2,"This drink is so ""super energy"" it's almost fr...",negative,this drink is so super energy it s almost frig...
2,2,"I'm sticking with what used to be carnation, n...",negative,i m sticking with what used to be carnation no...
3,1,Aspertame causes alot of problems including pr...,negative,aspertame causes alot of problems including pr...
4,2,I ordered these because my local pet store sto...,negative,i ordered these because my local pet store sto...


## 2. Lexicon-based Approach: VADER

In [23]:
# Initialize VADER
analyzer = SentimentIntensityAnalyzer()

# Apply VADER on the original Text (VADER works better with punctuation/caps)
def vader_sentiment(text):
    compound = analyzer.polarity_scores(text)['compound']
    if compound > 0.05:
        return 'positive'
    elif compound < -0.05:
        return 'negative'
    else:
        return 'neutral'

df_sampled['vader_sentiment'] = df_sampled['Text'].apply(vader_sentiment)
print('VADER prediction distribution:')
print(df_sampled['vader_sentiment'].value_counts())

VADER prediction distribution:
vader_sentiment
positive    69010
negative    17995
neutral      2995
Name: count, dtype: int64


In [9]:
# VADER Classification Report
print('Classification Report for VADER:')
print(classification_report(df_sampled['sentiment'], df_sampled['vader_sentiment'],
                            target_names=['negative', 'neutral', 'positive']))
print(f'VADER Accuracy: {accuracy_score(df_sampled["sentiment"], df_sampled["vader_sentiment"]):.4f}')

Classification Report for VADER:
              precision    recall  f1-score   support

    negative       0.66      0.40      0.50      5000
     neutral       0.37      0.04      0.07      5000
    positive       0.42      0.96      0.58      5000

    accuracy                           0.46     15000
   macro avg       0.48      0.46      0.38     15000
weighted avg       0.48      0.46      0.38     15000

VADER Accuracy: 0.4640


In [10]:
# VADER Confusion Matrix
print('Confusion Matrix for VADER:')
print(confusion_matrix(df_sampled['sentiment'], df_sampled['vader_sentiment'],
                       labels=['negative', 'neutral', 'positive']))

Confusion Matrix for VADER:
[[2002  254 2744]
 [ 854  180 3966]
 [ 165   57 4778]]


## 3. Machine Learning-based Approach: Naive Bayes

### Feature Extraction using TF-IDF

In [11]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    df_sampled['clean_text'], df_sampled['sentiment'],
    test_size=0.3, random_state=42, stratify=df_sampled['sentiment']
)

print(f'Training set: {len(X_train)} rows')
print(f'Test set: {len(X_test)} rows')

Training set: 10500 rows
Test set: 4500 rows


In [51]:
# TF-IDF Vectorization (limit features for speed)
tfidf = TfidfVectorizer(max_features=800000, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'TF-IDF matrix shape: {X_train_tfidf.shape}')

TF-IDF matrix shape: (10500, 274731)


In [52]:
# Train Naive Bayes
nb_classifier = MultinomialNB()
nb_classifier.fit(X_train_tfidf, y_train)

# Predict
nb_predictions = nb_classifier.predict(X_test_tfidf)

### Model Evaluation - Naive Bayes

In [53]:
# Naive Bayes Classification Report
print('Classification Report for Naive Bayes:')
print(classification_report(y_test, nb_predictions,
                            target_names=['negative', 'neutral', 'positive']))
print(f'Naive Bayes Accuracy: {accuracy_score(y_test, nb_predictions):.4f}')

Classification Report for Naive Bayes:
              precision    recall  f1-score   support

    negative       0.71      0.66      0.68      1500
     neutral       0.56      0.69      0.62      1500
    positive       0.82      0.68      0.74      1500

    accuracy                           0.68      4500
   macro avg       0.70      0.68      0.68      4500
weighted avg       0.70      0.68      0.68      4500

Naive Bayes Accuracy: 0.6780


In [54]:
# Naive Bayes Confusion Matrix
print('Confusion Matrix for Naive Bayes:')
print(confusion_matrix(y_test, nb_predictions,
                       labels=['negative', 'neutral', 'positive']))

Confusion Matrix for Naive Bayes:
[[ 984  447   69]
 [ 296 1042  162]
 [ 101  374 1025]]


## 4. Model Comparison

In [55]:
# Compare VADER vs Naive Bayes on the TEST set only
# Get VADER predictions for test set
vader_test_preds = X_test.map(vader_sentiment)

vader_acc = accuracy_score(y_test, vader_test_preds)
nb_acc = accuracy_score(y_test, nb_predictions)

print('=== Model Comparison (on Test Set) ===')
print(f'VADER Accuracy:       {vader_acc:.4f}')
print(f'Naive Bayes Accuracy: {nb_acc:.4f}')
print()
print('--- VADER (Test Set) ---')
print(classification_report(y_test, vader_test_preds, target_names=['negative', 'neutral', 'positive']))
print('--- Naive Bayes (Test Set) ---')
print(classification_report(y_test, nb_predictions, target_names=['negative', 'neutral', 'positive']))

=== Model Comparison (on Test Set) ===
VADER Accuracy:       0.4500
Naive Bayes Accuracy: 0.6780

--- VADER (Test Set) ---
              precision    recall  f1-score   support

    negative       0.68      0.36      0.47      1500
     neutral       0.32      0.03      0.06      1500
    positive       0.40      0.96      0.57      1500

    accuracy                           0.45      4500
   macro avg       0.47      0.45      0.37      4500
weighted avg       0.47      0.45      0.37      4500

--- Naive Bayes (Test Set) ---
              precision    recall  f1-score   support

    negative       0.71      0.66      0.68      1500
     neutral       0.56      0.69      0.62      1500
    positive       0.82      0.68      0.74      1500

    accuracy                           0.68      4500
   macro avg       0.70      0.68      0.68      4500
weighted avg       0.70      0.68      0.68      4500



## 5. Discussion

**VADER (Lexicon-based):**
- **Strengths:** Does not require training data, works out-of-the-box, and is fast to apply. It handles sentiment-laden words, punctuation, and capitalization well. Good at detecting clearly positive and negative reviews.
- **Weaknesses:** Struggles with neutral sentiment classification since its lexicon is designed to detect polarity. It cannot learn domain-specific patterns (e.g., food-related sentiment cues) and tends to misclassify neutral reviews as positive or negative.

**Naive Bayes (Machine Learning-based):**
- **Strengths:** Learns patterns from the training data, so it adapts to the specific language used in Amazon food reviews. With TF-IDF features, it captures important domain-specific words and achieves higher overall accuracy. It handles all three classes more effectively.
- **Weaknesses:** Requires labeled training data and may not generalize well to other domains without retraining. The bag-of-words assumption ignores word order and context. Performance depends heavily on the quality of text preprocessing and feature extraction.

**Overall:** Naive Bayes outperforms VADER on this dataset because it learns from the review data directly, while VADER relies on a general-purpose sentiment lexicon that is not tuned for food reviews. However, VADER is useful when labeled data is unavailable.

In [56]:
# Save the sampled/processed data as CSV for submission
df_sampled[['Text', 'Score', 'sentiment', 'clean_text']].to_csv('Reviews_sampled.csv', index=False)
print('Saved Reviews_sampled.csv')

Saved Reviews_sampled.csv
